## Part 2

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("mtsamples.csv", index_col = 0)
df.head(5)

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df['transcription'].head(5)

In [ ]:
df['medical_specialty'].head(5)

In [ ]:
df.isnull().sum()

In [ ]:
col = df.columns[df.isna().any()].tolist()
df[col] = df[col].fillna("")

In [ ]:
import re
import pandas as pd


# ---------------------------------------------------------
# PHI patterns
# ---------------------------------------------------------
patterns = {
    "DATE": r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}[/-]\d{1,2}[/-]\d{1,2})\b",

    "PHONE": r"\b(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b",

    "MRN": r"\b(?:MRN|MR#|Medical Record Number|Record Number)\s*[:#-]?\s*[A-Za-z0-9-]+\b",

    "EMAIL": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",

    "NAME": r"\b(?:Patient Name|Patient|Name)\s*[:\-]\s*[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*"
}


# ---------------------------------------------------------
# Find PHI
# Returns PHI type + actual matched value
# ---------------------------------------------------------
def find_phi(text):

    if pd.isna(text):
        return []

    text = str(text)

    findings = []

    for phi_type, pattern in patterns.items():

        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        for match in matches:
            findings.append((phi_type, match))

    return findings


# ---------------------------------------------------------
# Find PHI in the original transcription
# ---------------------------------------------------------
df["PHI_found"] = df["transcription"].apply(find_phi)


# ---------------------------------------------------------
# Redact PHI
# ---------------------------------------------------------
def redact_phi(text):

    if pd.isna(text):
        return ""

    text = str(text)

    for phi_type, pattern in patterns.items():

        text = re.sub(
            pattern,
            f"[{phi_type}]",
            text,
            flags=re.IGNORECASE
        )

    return text


# ---------------------------------------------------------
# Create redacted transcription column
# ---------------------------------------------------------
df["transcription_redacted"] = df["transcription"].apply(redact_phi)


# ---------------------------------------------------------
# Display rows where PHI was detected
# ---------------------------------------------------------
phi_rows = df[df["PHI_found"].str.len() > 0][
    ["transcription", "PHI_found", "transcription_redacted"]
]

print(phi_rows.head())


In [ ]:
df

In [ ]:
df.loc[df["PHI_found"].str.len() > 0, ["transcription", "PHI_found"]]

## Text Cleaning

In [ ]:
import re
from bs4 import BeautifulSoup

def clean_text(text):
    if not text:
        return ""
    # 1. Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()
    
    # 2. Remove section header noise (ALL-CAPS words followed by colon)
    text = re.sub(r'\b[A-Z\s]{2,}:', ' ', text)
    
    # 3. Remove non-alphanumeric characters (excluding spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    
    # 4. Remove extra whitespaces and lowercase
    text = re.sub(r'\s+', ' ', text).strip().lower()
    
    return text

df["cleaned_text"] = df["transcription_redacted"].apply(clean_text)
df[["transcription_redacted", "cleaned_text"]].head()

## Tokenization & Stop word Removal

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# nltk.download('nltk_data')
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Common medical abbreviation dictionary
medical_abbreviations = {
    "pt": "patient",
    "hx": "history",
    "dx": "diagnosis",
    "rx": "prescription",
    "sob": "shortness of breath",
    "bp": "blood pressure",
    "hr": "heart rate",
    "yo": "years old",
    "y/o": "years old",
    "cc": "chief complaint",
    "prn": "as needed",
    "po": "by mouth"
}

def process_and_normalize(text):
    tokens = word_tokenize(text)
    
    processed_tokens = []
    for token in tokens:
        
        # Expand abbreviation if present, else keep token
        expanded = medical_abbreviations.get(token, token)
        
        # Expand multi-word abbreviations if any
        for word in expanded.split():
            if word not in stop_words and len(word) > 1:
                # Lemmatize
                lemmatized = lemmatizer.lemmatize(word)
                processed_tokens.append(lemmatized)
                
    return processed_tokens

# Tokenize, normalize, and reconstitute processed text
df["tokens"] = df["cleaned_text"].apply(process_and_normalize)
df["normalized_text"] = df["tokens"].apply(lambda x: " ".join(x))
df[["cleaned_text", "normalized_text"]].head()

In [ ]:
df

## Feature Extraction 

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit and transform normalized text
tfidf_matrix = tfidf.fit_transform(df["normalized_text"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

NameError: name 'df' is not defined

In [ ]:
print(tfidf_matrix.toarray())